# Patient Recommender System

Using RAG framework to retrieve cases and provide summaries for patients that are of similar nature

| Date | User | Change Type | Remarks |  
| ---- | ---- | ----------- | ------- |
| 30/03/2026   | Martin | Create  | Notebook created. Started working on document format for patient profile | 
| 31/03/2026   | Martin | Update  | Format document for both MIMIC and Synthea data | 

# Content

* [Patient Snapshot](#patient-snapshot)
* [Idea for Document](#idea-for-document)

# Patient Snapshot

Explore the availability of data from both Synthea and MIMIC resources to craft the patient's profile

In [17]:
import duckdb

In [18]:
con = duckdb.connect("../data/warehouse/mimic_fhir.duckdb")

In [3]:
con.sql("SHOW TABLES FROM silver;")

┌───────────────────────────────────┐
│               name                │
│              varchar              │
├───────────────────────────────────┤
│ condition                         │
│ encounter                         │
│ location                          │
│ medication_dispense               │
│ obs_vitals                        │
│ organization                      │
│ patient                           │
│ procedure                         │
│ synthea_allergy_intolerance       │
│ synthea_careplan                  │
│ synthea_careteam                  │
│ synthea_condition                 │
│ synthea_encounter                 │
│ synthea_immunization              │
│ synthea_medication                │
│ synthea_medication_administration │
│ synthea_medication_request        │
│ synthea_observation               │
│ synthea_patient                   │
│ synthea_procedure                 │
├───────────────────────────────────┤
│              20 rows              │
└───────────

## MIMIC Data

Observations from patients:

1. f040bdf0-6951-5194-a400-fac7a247583c (15174162)
2. f7ed2ddf-9129-51d0-9bad-0d53665559db (15653234)
3. d4566ed7-5ee5-5cb1-bb88-b08f5b0a8c03 (15653252)

<u>Data Present</u>

| Column | Table | Type |  
| ------ | ----- | ---- |
| name | patient | categorical |
| gender | patient | categorical |
| birth_date | patient | date (YYYY-DD-MM) |
| race | patient | categorical |
| ethnicity | patient | categorical |
| marital_status | patient | categorical |

In [11]:
id1 = "f040bdf0-6951-5194-a400-fac7a247583c"
id2 = "f7ed2ddf-9129-51d0-9bad-0d53665559db"
id3 = "d4566ed7-5ee5-5cb1-bb88-b08f5b0a8c03"

In [14]:
con.sql(f"""
SELECT 
  c.icd_code,
  c.icd_name,
  e.start_timestamp,
  e.end_timestamp,
  e.admit_source,
  e.discharge_disposition
FROM silver.condition c
LEFT JOIN silver.encounter e ON c.encounter_id = e.resource_id
WHERE c.patient_id='{id1}';
""")

┌──────────┬─────────────────────────────────────────────────────────────────────────────────┬─────────────────────┬─────────────────────┬──────────────┬───────────────────────┐
│ icd_code │                                    icd_name                                     │   start_timestamp   │    end_timestamp    │ admit_source │ discharge_disposition │
│ varchar  │                                     varchar                                     │      timestamp      │      timestamp      │   varchar    │        varchar        │
├──────────┼─────────────────────────────────────────────────────────────────────────────────┼─────────────────────┼─────────────────────┼──────────────┼───────────────────────┤
│ R502     │ Drug induced fever                                                              │ 2136-04-19 02:12:00 │ 2136-04-19 10:10:00 │ WALK IN      │ HOME                  │
│ R0902    │ Hypoxemia                                                                       │ 2136-07-07 12:1

In [36]:
con.sql(f"SELECT * FROM silver.condition WHERE patient_id='{id1}';")

┌──────────────────────────────────────┬──────────┬─────────────────────────────────────────────────────────────────────────────────┬──────────────────────────────────────┬─────────────────────┬──────────────────────────────────────┐
│             resource_id              │ icd_code │                                    icd_name                                     │              patient_id              │ condition_category  │             encounter_id             │
│               varchar                │ varchar  │                                     varchar                                     │               varchar                │       varchar       │               varchar                │
├──────────────────────────────────────┼──────────┼─────────────────────────────────────────────────────────────────────────────────┼──────────────────────────────────────┼─────────────────────┼──────────────────────────────────────┤
│ ff4a1fc3-16ff-5f20-ac65-2f08531c7fd0 │ R0902    │ Hypoxemia   

In [45]:
con.sql(f"SELECT * FROM silver.obs_vitals WHERE patient_id='{id1}';")

┌──────────────────────────────────────┬──────────────────────────────────────┬──────────────────────────────────────┬─────────────────────┬────────────┬────────┬────────────────┐
│             resource_id              │              patient_id              │             encounter_id             │ effective_datetime  │ loinc_code │ value  │      unit      │
│               varchar                │               varchar                │               varchar                │      timestamp      │  varchar   │ double │    varchar     │
├──────────────────────────────────────┼──────────────────────────────────────┼──────────────────────────────────────┼─────────────────────┼────────────┼────────┼────────────────┤
│ 31cf70a6-5495-56b9-ac34-f4c193732530 │ f040bdf0-6951-5194-a400-fac7a247583c │ ef4161fd-b864-5f39-a1c9-aea9ba376086 │ 2136-07-07 12:14:00 │ 8867-4     │  145.0 │ beats/minute   │
│ 2ee943a9-29af-580f-b8d8-7007775c1882 │ f040bdf0-6951-5194-a400-fac7a247583c │ ef4161fd-b864-5f39-a

In [9]:
con.sql("SELECT DISTINCT discharge_disposition FROM silver.encounter;")

┌─────────────────────────────┐
│    discharge_disposition    │
│           varchar           │
├─────────────────────────────┤
│ HOME                        │
│ LEFT AGAINST MEDICAL ADVICE │
│ LEFT WITHOUT BEING SEEN     │
│ OTHER                       │
│ ADMITTED                    │
│ ELOPED                      │
│ EXPIRED                     │
│ TRANSFER                    │
└─────────────────────────────┘

### Transferring MIMIC silver to gold

Here we transform silver MIMIC data to gold tables

In [57]:
id1 = "7d101f5f-aba3-586c-94b2-6055c90992a2"

In [100]:
con.sql("SELECT table_name FROM information_schema.tables WHERE table_schema = 'temp'")

┌────────────┐
│ table_name │
│  varchar   │
├────────────┤
│ conditions │
│ patient    │
│ procedures │
└────────────┘

In [27]:
con.sql("DROP TABLE mimic_fhir.temp.patient;")

In [ ]:
# Want to copy the schema of the silver.patient table to first create
con.execute("""
CREATE TABLE IF NOT EXISTS mimic_fhir.temp.patient AS
SELECT
  *,
  0 AS documented
FROM silver.patient
LIMIT 0;
""")

con.execute("""
ALTER TABLE mimic_fhir.temp.patient
ALTER COLUMN documented SET DEFAULT 0;
""")

# Create the conditions table
con.execute("""
CREATE TABLE IF NOT EXISTS mimic_fhir.temp.conditions (
  patient_id VARCHAR,
  icd_code VARCHAR,
  icd_name VARCHAR,
  start_timestamp TIMESTAMP,
  end_timestamp TIMESTAMP
)
""")

# Create the procedures table
con.execute("""
CREATE TABLE IF NOT EXISTS mimic_fhir.temp.procedures (
  patient_id VARCHAR,
  performed_datetime TIMESTAMP,
  snomed_ct_id VARCHAR,
  snomed_ct_procedure VARCHAR
)
""")

In [34]:
# On initial insert will take all patients
con.execute("""
INSERT INTO mimic_fhir.temp.patient
SELECT *, 0 AS documented
FROM silver.patient
LIMIT 20;
""")

In [46]:
# On subsequent it inserts it will compare the current patient id with the silver tables ones - This should be a pipeline script
con.execute("""
INSERT INTO mimic_fhir.temp.patient
SELECT
  t1.*,
  0 AS documented
FROM silver.patient t1
WHERE NOT EXISTS (
  SELECT 1 FROM mimic_fhir.temp.patient t2 WHERE t1.resource_id = t2.resource_id
);
""")

In [ ]:
con.sql(f"""
        INSERT INTO mimic_fhir.temp.conditions
        SELECT 
                c.patient_id,
                c.icd_code,
                c.icd_name,
                e.start_timestamp,
                e.end_timestamp
        FROM silver.condition c
        LEFT JOIN silver.encounter e ON c.encounter_id = e.resource_id;
        ;""")

con.sql(f"""
        INSERT INTO mimic_fhir.temp.procedures
        SELECT 
          patient_id,
          performed_datetime,
          snomed_ct_id,
          snomed_ct_procedure
        FROM silver.procedure;
        ;""")

In [103]:
con.sql("SELECT COUNT(1) FROM mimic_fhir.temp.procedures;")

┌──────────┐
│ count(1) │
│  int64   │
├──────────┤
│  1989697 │
└──────────┘

Transform into documents

In [78]:
from langchain_core.documents import Document
from datetime import datetime

In [71]:
patient = con.sql("SELECT * FROM mimic_fhir.temp.patient WHERE documented = 0 LIMIT 1;").fetchnumpy()
patient = {k: v[0] for k, v in patient.items()}
patient

{'resource_id': '7d101f5f-aba3-586c-94b2-6055c90992a2',
 'name': 'Patient_15173979',
 'gender': 'male',
 'birth_date': '2072-02-02',
 'race': 'unknown',
 'ethnicity': masked,
 'patient_num': '15173979',
 'language': 'en',
 'marital_status': 'S',
 'organization_id': 'ee172322-118b-5716-abbc-18e4c5437e15',
 'documented': np.int32(0)}

In [107]:
condition = con.sql(f"SELECT * FROM mimic_fhir.temp.conditions WHERE patient_id = '{patient['resource_id']}';").fetchnumpy()
condition

{'patient_id': array(['7d101f5f-aba3-586c-94b2-6055c90992a2',
        '7d101f5f-aba3-586c-94b2-6055c90992a2',
        '7d101f5f-aba3-586c-94b2-6055c90992a2',
        '7d101f5f-aba3-586c-94b2-6055c90992a2',
        '7d101f5f-aba3-586c-94b2-6055c90992a2',
        '7d101f5f-aba3-586c-94b2-6055c90992a2'], dtype=object),
 'icd_code': array(['8028', 'E9650', '80156', '87341', '87202', '37923'], dtype=object),
 'icd_name': array(['FX FACIAL BONE NEC-CLOSE', 'Assault by handgun',
        'Open fracture of base of skull without mention of intracranial injury, with loss of consciousness of unspecified duration',
        'Open wound of cheek, without mention of complication',
        'Open wound of auditory canal, without mention of complication',
        'Vitreous hemorrhage'], dtype=object),
 'start_timestamp': array(['2110-02-02T16:35:00.000000', '2110-02-02T16:35:00.000000',
        '2110-02-02T16:35:00.000000', '2110-02-02T16:35:00.000000',
        '2110-02-02T16:35:00.000000', '2110-02-02T1

In [106]:
procedures = con.sql(f"SELECT * FROM mimic_fhir.temp.procedures WHERE patient_id = '{patient['resource_id']}';").fetchnumpy()
procedures

{'patient_id': array(['7d101f5f-aba3-586c-94b2-6055c90992a2'], dtype=object),
 'performed_datetime': array(['2110-02-02T16:35:00.000000'], dtype='datetime64[us]'),
 'snomed_ct_id': array(['386478007'], dtype=object),
 'snomed_ct_procedure': array(['Triage: emergency center'], dtype=object)}

In [109]:
def create_mimic_patient_document(
  patient_dict: dict,
  conditions_dict: dict,
  procedures_dict: dict,
) -> Document:
  # Initial patient info
  page_content = f"""
Patient Information:
- Name: {patient_dict.get('name', '')}
- Gender: {patient_dict.get('gender', '')}
- DOB: {patient_dict.get('birth_date', '')}
- Race: {patient_dict.get('race', '')}
- Ethnicity: {patient_dict.get('ethnicity', '')}
- Marital Status: {patient_dict.get('marital_status', '')}
  """

  # Conditions info
  base_conditions = """
Conditions
ONSET DATE, DIAGNOSIS DATE, CODE, CONDITION
"""
  for i in range(len(conditions_dict['patient_id'])):
    start_time = conditions_dict['start_timestamp'][i].astype('datetime64[s]').item().strftime('%m-%d-%Y %H:%M:%S')
    end_time = conditions_dict['end_timestamp'][i].astype('datetime64[s]').item().strftime('%m-%d-%Y %H:%M:%S')
    base_conditions += f"{i+1}. {start_time}, {end_time}, {conditions_dict['icd_code'][i]}, {conditions_dict['icd_name'][i]}\n"
  page_content += f"\n{base_conditions}"

  # Procedures info
  base_procedures = """
Procedures
PERFORMED DATE, CODE, PROCEDURE
"""
  for i in range(len(procedures_dict['patient_id'])):
    procedure_time = procedures_dict['performed_datetime'][i].astype('datetime64[s]').item().strftime('%m-%d-%Y %H:%M:%S')
    base_procedures += f"{i+1}. {procedure_time}, {procedures_dict['snomed_ct_id'][0]}, {procedures_dict['snomed_ct_procedure'][0]}"
  page_content += f"\n{base_procedures}"

  return page_content


In [110]:
print(create_mimic_patient_document(patient, condition, procedures))


Patient Information:
- Name: Patient_15173979
- Gender: male
- DOB: 2072-02-02
- Race: unknown
- Ethnicity: --
- Marital Status: S
  

Conditions
ONSET DATE, DIAGNOSIS DATE, CODE, CONDITION
1. 02-02-2110 16:35:00, 02-02-2110 18:56:35, 8028, FX FACIAL BONE NEC-CLOSE
2. 02-02-2110 16:35:00, 02-02-2110 18:56:35, E9650, Assault by handgun
3. 02-02-2110 16:35:00, 02-02-2110 18:56:35, 80156, Open fracture of base of skull without mention of intracranial injury, with loss of consciousness of unspecified duration
4. 02-02-2110 16:35:00, 02-02-2110 18:56:35, 87341, Open wound of cheek, without mention of complication
5. 02-02-2110 16:35:00, 02-02-2110 18:56:35, 87202, Open wound of auditory canal, without mention of complication
6. 02-02-2110 16:35:00, 02-02-2110 18:56:35, 37923, Vitreous hemorrhage


Procedures
PERFORMED DATE, CODE, PROCEDURE
1. 02-02-2110 16:35:00, 386478007, Triage: emergency center


In [16]:
con.close()

## Synthea Data

Observations from patients:

1. 24283ca3-2f37-a4c1-e740-97f060a230a6
2. 2e312e34-cc56-6a67-6642-c261c2f4838f
3. a3d61946-69d4-cd45-0792-f8b941bd6f73

<u>Data Present</u>

| Column | Table | Type | Notes | 
| ------ | ----- | ---- | ----- |
| name | patient | categorical | Combine first_name + middle_name + last_name |
| gender | patient | categorical | |
| birth_date | patient | date (YYYY-DD-MM) | |
| race | patient | categorical | |
| ethnicity | patient | categorical | |
| marital_status | patient | categorical | Convert to MIMIC format |


In [4]:
syn_id1 = "24283ca3-2f37-a4c1-e740-97f060a230a6"
syn_id2 = "2e312e34-cc56-6a67-6642-c261c2f4838f"
syn_id3 = "a3d61946-69d4-cd45-0792-f8b941bd6f73"

In [13]:
con.sql("SELECT * FROM silver.synthea_patient;")

┌──────────────────────────────────────┬─────────────┬─────────────┬────────────┬─────────┬───────────────────────────────────────────┬────────────────────────┬────────────┬──────────────┬────────────────┬──────────────────┬─────────┬─────────┬───────────────────────────────────────┬────────────────────┬────────────────────┬─────────────────────────────────┬─────────────────────────────┐
│             resource_id              │ first_name  │ middle_name │ last_name  │ gender  │                   race                    │       ethnicity        │ birth_date │ phone_number │ marital_status │       city       │  state  │ country │            street_address             │      latitude      │     longitude      │ disablility_adjusted_life_years │ quality_adjusted_life_years │
│               varchar                │   varchar   │   varchar   │  varchar   │ varchar │                  varchar                  │        varchar         │  varchar   │   varchar    │    varchar     │     varchar 

In [44]:
con.sql(f"""
SELECT 
  c.condition_code,
  c.condition,
  c.condition_type,
  c.patient_id,
  c.onset_timestamp,
  c.recorded_timestamp,
  e.procedure_code,
  e.procedure_name,
  m.medication_code,
  m.medication,
  m.effective_timestamp,
  m.reason_id,
  m.reason,
  m.reason_type,
FROM silver.synthea_condition c
LEFT JOIN silver.synthea_encounter e ON c.encounter_id = e.resource_id
LEFT JOIN silver.synthea_medication_administration m ON c.encounter_id = m.context_id
WHERE c.patient_id='{syn_id2}';
""")

┌────────────────┬───────────────────────────────────────┬────────────────┬──────────────────────────────────────┬─────────────────────┬─────────────────────┬────────────────┬─────────────────────────────────┬─────────────────┬───────────────────────────────────────┬─────────────────────┬──────────────────────────────────────┬───────────────────────────────────┬─────────────┐
│ condition_code │               condition               │ condition_type │              patient_id              │   onset_timestamp   │ recorded_timestamp  │ procedure_code │         procedure_name          │ medication_code │              medication               │ effective_timestamp │              reason_id               │              reason               │ reason_type │
│    varchar     │                varchar                │    varchar     │               varchar                │      timestamp      │      timestamp      │    varchar     │             varchar             │     varchar     │               

In [41]:
con.sql("SELECT * FROM silver.synthea_encounter LIMIT 10;")

┌──────────────────────────────────────┬──────────┬─────────┬────────────────┬────────────────────────────────┬──────────────────────────────────────┬───────────────────┬───────────────────┬─────────────────┬──────────────────────────────┬─────────────────────┬─────────────────────┬──────────────────────────────────────┬────────────────────────────────────┬──────────────────────────────────────┬────────────────────────────────────┐
│             resource_id              │  status  │  class  │ procedure_code │         procedure_name         │              patient_id              │ practitioner_code │ practitioner_type │ practitioner_id │      practitioner_name       │     start_date      │      end_date       │             location_id              │           location_name            │             provider_id              │           provider_name            │
│               varchar                │ varchar  │ varchar │    varchar     │            varchar             │               va

In [34]:
con.sql(f"SELECT * FROM silver.synthea_medication_administration WHERE patient_id='{syn_id2}' LIMIT 10;")

┌──────────────────────────────────────┬───────────┬─────────────────┬───────────────────────────────────────┬──────────────────────────────────────┬──────────────────────────────────────┬─────────────────────┬──────────────────────────────────────┬───────────────────────────────────┬─────────────┐
│             resource_id              │  status   │ medication_code │              medication               │              patient_id              │              context_id              │ effective_timestamp │              reason_id               │              reason               │ reason_type │
│               varchar                │  varchar  │     varchar     │                varchar                │               varchar                │               varchar                │      timestamp      │               varchar                │              varchar              │   varchar   │
├──────────────────────────────────────┼───────────┼─────────────────┼──────────────────────────────

In [46]:
con.sql(f"SELECT * FROM silver.synthea_observation WHERE patient_id='{syn_id2}';")

┌──────────────────────────────────────┬─────────┬────────────────┬──────────────┬─────────────────────────────────────────────────────────────────────────────────────────────┬─────────┬─────────┬──────────────────────────────────────┬─────────────────────┬─────────────────────────┐
│             resource_id              │ status  │     record     │ measure_code │                                        measure_name                                         │  value  │  unit   │              patient_id              │ effective_timestamp │    issued_timestamp     │
│               varchar                │ varchar │    varchar     │   varchar    │                                           varchar                                           │ double  │ varchar │               varchar                │      timestamp      │        timestamp        │
├──────────────────────────────────────┼─────────┼────────────────┼──────────────┼──────────────────────────────────────────────────────────────────

# Idea for Document

Patient Information
- Name: {name}
- Gender: {gender}
- Birth Date: {birth_date}
- Race: {race}
- Ethnicity: {ethnicity}
- Marital Status: {marital_status}

Conditions
| Onset Date | Diagnosis Date | ICD-10 Code | Condition |
| ---------- | -------------- | ----------- | --------- |
| {start_date} | {end_date} | {icd_10_code} | {condition_name} 
...

Procedures
| Performed Date |  snomed_ct Code | Procedure |
| -------------- |  ----------- | --------- |
| {performed_date} | {procedure_code} | {procedure} |
...

Medication Dispensed
| Administered Date | Medication Code | Medication |
| ----------------- | --------------- | ---------- |
| {effective_timestamp} | {medication_code} | {medication} |
...

Vitals (Observation)
| Observation Date | Measured Vitals |
| ---------------- | --------------- |
| {effective_timestamp} | {vitals_name}: {value}{unit}, {vitals_name}: {value}{unit} |
...



In [2]:
%watermark

Last updated: 2025-06-18T19:03:45.452311+08:00

Python implementation: CPython
Python version       : 3.11.9
IPython version      : 8.31.0

Compiler    : MSC v.1938 64 bit (AMD64)
OS          : Windows
Release     : 10
Machine     : AMD64
Processor   : Intel64 Family 6 Model 183 Stepping 1, GenuineIntel
CPU cores   : 20
Architecture: 64bit

